# ПЗ 1 — Обработка изображений с OpenCV

**Задача:** загрузить изображение, применить базовые преобразования OpenCV, распознать числовой текст через EasyOCR.

**Стек:** `opencv-python`, `easyocr`, `numpy`, `matplotlib`, `pandas`

In [1]:
# Установка зависимостей (только в Colab)
!pip install opencv-python-headless easyocr pandas matplotlib -q

  error: subprocess-exited-with-error
  
  × Building wheel for python-bidi (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [53 lines of output]
      Rust not found, installing into a temporary directory
      Python reports SOABI: cp314-win_amd64
      Computed rustc target triple: x86_64-pc-windows-msvc
      Installation directory: C:\Users\User\AppData\Local\puccinialin\puccinialin\Cache
      Rustup already downloaded
      Installing rust to C:\Users\User\AppData\Local\puccinialin\puccinialin\Cache\rustup
      warn: It looks like you have an existing rustup settings file at:
      warn: C:\Users\User\AppData\Local\puccinialin\puccinialin\Cache\rustup\settings.toml
      warn: Rustup will install the default toolchain as specified in the settings file,
      warn: instead of the one inferred from the default host triple.
      warn: installing msvc toolchain without its prerequisites
      info: profile set to minimal
      info: setting default host triple to 

In [ ]:
# Загрузка тестового изображения
from google.colab import files
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
print(f'Загружен файл: {file_path}')

In [ ]:
import cv2
import numpy as np
import easyocr
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

reader = easyocr.Reader(['ru', 'en'], gpu=True)

img = cv2.imread(file_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

blurred = cv2.GaussianBlur(gray, (5, 5), 0)

sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_combined = cv2.magnitude(sobelx, sobely)

_, mask = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

R, G, B = cv2.split(img_rgb)

In [ ]:
plt.figure(figsize=(20, 12))
display_list = [
    (img_rgb,        'оригинал'),
    (gray,           'ч/б'),
    (blurred,        'сглаживание'),
    (sobel_combined, 'Собель (контуры)'),
    (mask,           'маска (порог)'),
    (R,              'канал R'),
    (G,              'канал G'),
    (B,              'канал B'),
]
for i, (image, title) in enumerate(display_list):
    plt.subplot(2, 4, i + 1)
    cmap = 'gray' if len(image.shape) == 2 else None
    plt.imshow(image, cmap=cmap)
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# OCR — распознавание числового текста
results = reader.readtext(gray)
final_data = []

for (bbox, text, prob) in results:
    clean_text = ''.join(filter(str.isdigit, text))
    if len(clean_text) >= 5:
        final_data.append({
            'изображение': file_path.split('/')[-1],
            'результат': clean_text,
            'уверенность': round(prob, 3),
        })
        break

if final_data:
    display(pd.DataFrame(final_data))
else:
    print('Числовой текст (5+ цифр) не найден')